In [45]:
"""
고양이 전문 분양 사이트를 Selenium으로 동적 크롤링하는 모듈입니다.

여러 TARGET_URL에 접속하고,
각 사이트의 페이지네이션을 순회하면서
페이지별 원본 HTML을 저장합니다.

저장 구조:
    data/raw/html/YYYYMMDD_HHMMSS/
        site_01_page_001.html
        site_01_page_002.html
        ...
        site_02_page_001.html
        ...

반환값:
    run_crawling()
        생성된 raw HTML 배치 폴더 경로
"""

from datetime import datetime
from pathlib import Path
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
)

In [94]:
## ===========================================================
## 1. 수집 설정
## ===========================================================

TARGET_URLS = [
    'https://www.dalunacats.com/product/list.php',
    'https://dogmaru.co.kr/356',
    'https://nebalhouse.com/cat/sub02.html?PHPSESSID=01cf106e68fafcb29071361a03ac0b99',
    'https://doremicat.co.kr/product/list.php',
]

WAIT_TIMEOUT = 10

PAGE_INTERVAL = 1

HEADLESS = False

## ===========================================================
## 2. 기본 저장 경로 설정
## ===========================================================

PROJECT_DIR = Path.cwd().resolve().parents[1] / 'anaconda'

RAW_HTML_DIR = PROJECT_DIR / 'data' / 'raw' / 'html'

print(f'프로젝트 기준 경로 : {PROJECT_DIR}')
print(f'원본 HTML 기본 저장 경로 : {RAW_HTML_DIR}')

프로젝트 기준 경로 : D:\anaconda
원본 HTML 기본 저장 경로 : D:\anaconda\data\raw\html


In [48]:
def create_driver(
    headless: bool = HEADLESS,
) -> webdriver.Chrome:
    """
    Chrome WebDriver를 생성한다.

    Args:
        headless:
            브라우저 화면 표시 여부

    Returns:
        생성된 Chrome WebDriver
    """

    options = Options()

    if headless:
        options.add_argument('--headless=new')

    options.add_argument('--start-maximized')

    driver = webdriver.Chrome(
        options=options
    )

    return driver

In [50]:
driver = create_driver()

In [88]:
driver.get(TARGET_URLS[0])

In [52]:
print(driver.title)
print(driver.current_url)

달루나캣츠🌙 24시 고양이분양전문 • 서울고양이분양 • 마포고양이분양 🐈‍⚩ 캐터리 출신 부모묘 확인OK
https://www.dalunacats.com/product/list.php


In [53]:
CAT_CARD_SELECTOR = 'ul.cat_list li'

wait = WebDriverWait(
    driver,
    WAIT_TIMEOUT
)

cat_cards = wait.until(
    EC.presence_of_all_elements_located(
        (
            By.CSS_SELECTOR,
            CAT_CARD_SELECTOR
        )
    )
)

print(f'현재 페이지 고양이 카드 수 : {len(cat_cards)}')

현재 페이지 고양이 카드 수 : 20


In [54]:
def ensure_directory(
    directory: Path,
) -> Path:
    """
    지정한 폴더가 없으면 생성하고
    폴더 경로를 반환한다.
    """

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    return directory

In [55]:
def create_batch_directory(
    directory: Path,
    collected_at: datetime,
) -> Path:
    """
    수집 시작 시각을 이름으로 사용하는
    배치 폴더를 생성한다.
    """

    batch_name = collected_at.strftime(
        '%Y%m%d_%H%M%S'
    )

    batch_dir = directory / batch_name

    return ensure_directory(
        batch_dir
    )

In [108]:
def save_raw_html(
    html: str,
    batch_dir: Path,
    site_no: int,
    source_page: int,
) -> Path:
    """
    Selenium으로 수집한 페이지 HTML을
    배치 폴더에 저장한다.
    """

    file_path = (
        batch_dir
        / f'site_{site_no:02d}_page_{source_page:03d}.html'
    )

    file_path.write_text(
        html,
        encoding='utf-8',
    )

    return file_path

In [109]:
def save_detail_html(
    html: str,
    batch_dir: Path,
    site_no: int,
    detail_no: int,
) -> Path:
    file_path = (
        batch_dir
        / f'site_{site_no:02d}_detail_{detail_no:04d}.html'
    )

    file_path.write_text(
        html,
        encoding='utf-8'
    )

    return file_path

In [97]:
def load_all_cats(
    driver: webdriver.Chrome,
    batch_dir: Path,
    site_no: int,
):
    wait = WebDriverWait(driver, WAIT_TIMEOUT)

    current_page = 1

    while True:
        try:
            cat_cards = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, CAT_CARD_SELECTOR)
                )
            )

            print(
                f'{site_no}번 사이트 '
                f'{current_page}페이지 '
                f'고양이 카드 수 : {len(cat_cards)}'
            )

            save_raw_html(
                html=driver.page_source,
                batch_dir=batch_dir,
                site_no=site_no,
                source_page=current_page,
            )

            first_card = cat_cards[0]

            next_page = current_page + 1

            try:
                next_button = driver.find_element(
                    By.XPATH,
                    f'//div[contains(@class, "paginate")]'
                    f'//a[normalize-space()="{next_page}"]'
                )

            except NoSuchElementException:
                next_button = driver.find_element(
                    By.XPATH,
                    '//div[contains(@class, "paginate")]'
                    '//img[contains(@src, "/images/paging/right.jpg")]'
                    '/ancestor::a[1]'
                )

            next_button.click()

            wait.until(
                EC.staleness_of(first_card)
            )

            current_page = next_page

            time.sleep(PAGE_INTERVAL)

        except (TimeoutException, NoSuchElementException):
            print(
                f'{site_no}번 사이트 '
                f'{current_page}페이지에서 수집 종료'
            )
            break

In [104]:
collected_at = datetime.now()

batch_dir = create_batch_directory(
    RAW_HTML_DIR,
    collected_at,
)

print(batch_dir)

D:\anaconda\data\raw\html\20260827_233751


# 1번째 사이트 HTML 수집    

In [120]:
driver.get(TARGET_URLS[0])

In [110]:
load_all_cats(
    driver=driver,
    batch_dir=batch_dir,
    site_no=1,
)

1번 사이트 1페이지 고양이 카드 수 : 20
1번 사이트 2페이지 고양이 카드 수 : 20
1번 사이트 3페이지 고양이 카드 수 : 20


KeyboardInterrupt: 

In [117]:
driver.get(TARGET_URLS[0])

wait = WebDriverWait(
    driver,
    WAIT_TIMEOUT
)

cat_cards = wait.until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, CAT_CARD_SELECTOR)
    )
)

detail_urls = []

for card in cat_cards:
    link_tag = card.find_element(
        By.CSS_SELECTOR,
        'a'
    )

    detail_urls.append(
        link_tag.get_attribute('href')
    )

print(
    f'1페이지 상세 URL 수 : {len(detail_urls)}'
)

1페이지 상세 URL 수 : 20


In [128]:
driver.get(TARGET_URLS[0])

wait = WebDriverWait(
    driver,
    WAIT_TIMEOUT
)

current_page = 1
detail_urls = []

while True:
    try:
        cat_cards = wait.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, CAT_CARD_SELECTOR)
            )
        )

        print(
            f'1번 사이트 {current_page}페이지 '
            f'카드 수 : {len(cat_cards)}'
        )

        for card in cat_cards:
            link_tag = card.find_element(
                By.CSS_SELECTOR,
                'a'
            )

            detail_url = link_tag.get_attribute(
                'href'
            )

            detail_urls.append(
                detail_url
            )

        first_card = cat_cards[0]
        next_page = current_page + 1

        try:
            next_button = driver.find_element(
                By.XPATH,
                f'//div[contains(@class, "paginate")]'
                f'//a[normalize-space()="{next_page}"]'
            )

        except NoSuchElementException:
            next_button = driver.find_element(
                By.XPATH,
                '//div[contains(@class, "paginate")]'
                '//img[contains(@src, "/images/paging/right.jpg")]'
                '/ancestor::a[1]'
            )

        next_button.click()

        wait.until(
            EC.staleness_of(first_card)
        )

        current_page = next_page

        time.sleep(PAGE_INTERVAL)

    except (TimeoutException, NoSuchElementException):
        print(
            f'{current_page}페이지에서 '
            f'목록 순회 종료'
        )
        break

1번 사이트 1페이지 카드 수 : 20
1번 사이트 2페이지 카드 수 : 20
1번 사이트 3페이지 카드 수 : 20
1번 사이트 4페이지 카드 수 : 20
1번 사이트 5페이지 카드 수 : 20
1번 사이트 6페이지 카드 수 : 20
1번 사이트 7페이지 카드 수 : 20
1번 사이트 8페이지 카드 수 : 20
1번 사이트 9페이지 카드 수 : 20
1번 사이트 10페이지 카드 수 : 20
1번 사이트 11페이지 카드 수 : 20
1번 사이트 12페이지 카드 수 : 20
1번 사이트 13페이지 카드 수 : 20
1번 사이트 14페이지 카드 수 : 20
1번 사이트 15페이지 카드 수 : 20
1번 사이트 16페이지 카드 수 : 20
1번 사이트 17페이지 카드 수 : 20
1번 사이트 18페이지 카드 수 : 20
1번 사이트 19페이지 카드 수 : 20
1번 사이트 20페이지 카드 수 : 20
1번 사이트 21페이지 카드 수 : 20
1번 사이트 22페이지 카드 수 : 20
1번 사이트 23페이지 카드 수 : 20
1번 사이트 24페이지 카드 수 : 20
1번 사이트 25페이지 카드 수 : 20
1번 사이트 26페이지 카드 수 : 20
1번 사이트 27페이지 카드 수 : 20
1번 사이트 28페이지 카드 수 : 20
1번 사이트 29페이지 카드 수 : 20
1번 사이트 30페이지 카드 수 : 20
1번 사이트 31페이지 카드 수 : 20
1번 사이트 32페이지 카드 수 : 20
1번 사이트 33페이지 카드 수 : 20
1번 사이트 34페이지 카드 수 : 20
1번 사이트 35페이지 카드 수 : 20
1번 사이트 36페이지 카드 수 : 20
1번 사이트 37페이지 카드 수 : 20
1번 사이트 38페이지 카드 수 : 20
1번 사이트 39페이지 카드 수 : 20
1번 사이트 40페이지 카드 수 : 20
1번 사이트 41페이지 카드 수 : 20
1번 사이트 42페이지 카드 수 : 20
1번 사이트 43페이지 카드 수 : 20
1번 사이트 44페이지 카드 수 : 

In [129]:
unique_detail_urls = list(
    dict.fromkeys(detail_urls)
)

for detail_no, detail_url in enumerate(
    unique_detail_urls[20:],
    start=21,
):
    driver.get(detail_url)

    wait.until(
        EC.presence_of_element_located(
            (
                By.CSS_SELECTOR,
                'ul.cat_spec'
            )
        )
    )

    save_detail_html(
        html=driver.page_source,
        batch_dir=batch_dir,
        site_no=1,
        detail_no=detail_no,
    )

    print(
        f'{detail_no}번째 상세페이지 저장 완료'
    )

    time.sleep(PAGE_INTERVAL)

21번째 상세페이지 저장 완료
22번째 상세페이지 저장 완료
23번째 상세페이지 저장 완료
24번째 상세페이지 저장 완료
25번째 상세페이지 저장 완료
26번째 상세페이지 저장 완료
27번째 상세페이지 저장 완료
28번째 상세페이지 저장 완료
29번째 상세페이지 저장 완료
30번째 상세페이지 저장 완료
31번째 상세페이지 저장 완료
32번째 상세페이지 저장 완료
33번째 상세페이지 저장 완료
34번째 상세페이지 저장 완료
35번째 상세페이지 저장 완료
36번째 상세페이지 저장 완료
37번째 상세페이지 저장 완료
38번째 상세페이지 저장 완료
39번째 상세페이지 저장 완료
40번째 상세페이지 저장 완료
41번째 상세페이지 저장 완료
42번째 상세페이지 저장 완료
43번째 상세페이지 저장 완료
44번째 상세페이지 저장 완료
45번째 상세페이지 저장 완료
46번째 상세페이지 저장 완료
47번째 상세페이지 저장 완료
48번째 상세페이지 저장 완료
49번째 상세페이지 저장 완료
50번째 상세페이지 저장 완료
51번째 상세페이지 저장 완료
52번째 상세페이지 저장 완료
53번째 상세페이지 저장 완료
54번째 상세페이지 저장 완료
55번째 상세페이지 저장 완료
56번째 상세페이지 저장 완료
57번째 상세페이지 저장 완료
58번째 상세페이지 저장 완료
59번째 상세페이지 저장 완료
60번째 상세페이지 저장 완료
61번째 상세페이지 저장 완료
62번째 상세페이지 저장 완료
63번째 상세페이지 저장 완료
64번째 상세페이지 저장 완료
65번째 상세페이지 저장 완료
66번째 상세페이지 저장 완료
67번째 상세페이지 저장 완료
68번째 상세페이지 저장 완료
69번째 상세페이지 저장 완료
70번째 상세페이지 저장 완료
71번째 상세페이지 저장 완료
72번째 상세페이지 저장 완료
73번째 상세페이지 저장 완료
74번째 상세페이지 저장 완료
75번째 상세페이지 저장 완료
76번째 상세페이지 저장 완료
77번째 상세페이지 저장 완료
78번째 상세페이지 저장 완료
79번째 상세페이지 저장 

In [126]:
save_detail_html(
    html=driver.page_source,
    batch_dir=batch_dir,
    site_no=1,
    detail_no=1,
)

WindowsPath('D:/anaconda/data/raw/html/20260827_233751/site_01_detail_0001.html')

In [127]:
print(len(unique_detail_urls))

120


# 2번째 사이트 HTML 수집

In [119]:
driver.get(TARGET_URLS[1])

SITE_02_CARD_SELECTOR = 'div.list-style-card._card_wrap'

cat_cards = wait.until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, SITE_02_CARD_SELECTOR)
    )
)

print(f'2번 사이트 고양이 카드 수 : {len(cat_cards)}')

2번 사이트 고양이 카드 수 : 80


In [99]:
save_raw_html(
    html=driver.page_source,
    batch_dir=batch_dir,
    site_no=2,
    source_page=1,
)

WindowsPath('D:/anaconda/data/raw/html/20260827_232246/site_02_page_001.html')

# 3번째 사이트 HTML 수집

In [102]:
driver.get(TARGET_URLS[2])

In [103]:
SITE_03_CARD_SELECTOR = 'ul.list-adopt li'
current_page = 1

while True:
    try:
        cat_cards = wait.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, SITE_03_CARD_SELECTOR)
            )
        )

        print(
            f'3번 사이트 '
            f'{current_page}페이지 '
            f'고양이 카드 수 : {len(cat_cards)}'
        )

        save_raw_html(
            html=driver.page_source,
            batch_dir=batch_dir,
            site_no=3,
            source_page=current_page,
        )

        next_page = current_page + 1

        next_button = driver.find_element(
            By.XPATH,
            f'//a[contains(@class, "ng_bbs_page") '
            f'and normalize-space()="{next_page}"]'
        )

        first_card = cat_cards[0]

        next_button.click()

        wait.until(
            EC.staleness_of(first_card)
        )

        current_page = next_page

        time.sleep(PAGE_INTERVAL)

    except (NoSuchElementException, TimeoutException):
        print(
            f'3번 사이트 '
            f'{current_page}페이지에서 수집 종료'
        )
        break

3번 사이트 1페이지 고양이 카드 수 : 32
3번 사이트 2페이지 고양이 카드 수 : 32
3번 사이트 3페이지 고양이 카드 수 : 32
3번 사이트 4페이지 고양이 카드 수 : 32
3번 사이트 5페이지 고양이 카드 수 : 32
3번 사이트 6페이지 고양이 카드 수 : 32
3번 사이트 7페이지 고양이 카드 수 : 11
3번 사이트 7페이지에서 수집 종료


# 4번째 사이트 HTML 수집

In [105]:
driver.get(TARGET_URLS[3])

In [106]:
SITE_04_CARD_SELECTOR = 'ul.cats li'

current_page = 1

while True:
    try:
        cat_cards = wait.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, SITE_04_CARD_SELECTOR)
            )
        )

        print(
            f'4번 사이트 '
            f'{current_page}페이지 '
            f'고양이 카드 수 : {len(cat_cards)}'
        )

        save_raw_html(
            html=driver.page_source,
            batch_dir=batch_dir,
            site_no=4,
            source_page=current_page,
        )

        first_card = cat_cards[0]
        next_page = current_page + 1

        try:
            # 현재 페이지 그룹 안에 다음 숫자 페이지가 있으면 클릭
            next_button = driver.find_element(
                By.XPATH,
                f'//ul[contains(@class, "paging")]'
                f'//a[normalize-space()="{next_page}"]'
            )

        except NoSuchElementException:
            # 다음 숫자 페이지가 없으면 다음 10페이지 그룹 버튼 클릭
            next_button = driver.find_element(
                By.XPATH,
                '//ul[contains(@class, "paging")]'
                '//img[contains(@src, "/images/sub/next.png")]'
                '/ancestor::a[1]'
            )

        next_button.click()

        wait.until(
            EC.staleness_of(first_card)
        )

        current_page = next_page

        time.sleep(PAGE_INTERVAL)

    except (TimeoutException, NoSuchElementException):
        print(
            f'4번 사이트 '
            f'{current_page}페이지에서 수집 종료'
        )
        break

4번 사이트 1페이지 고양이 카드 수 : 12
4번 사이트 2페이지 고양이 카드 수 : 12
4번 사이트 3페이지 고양이 카드 수 : 12
4번 사이트 4페이지 고양이 카드 수 : 12
4번 사이트 5페이지 고양이 카드 수 : 12
4번 사이트 6페이지 고양이 카드 수 : 12
4번 사이트 7페이지 고양이 카드 수 : 12
4번 사이트 8페이지 고양이 카드 수 : 12
4번 사이트 9페이지 고양이 카드 수 : 12
4번 사이트 10페이지 고양이 카드 수 : 12
4번 사이트 11페이지 고양이 카드 수 : 12
4번 사이트 12페이지 고양이 카드 수 : 12
4번 사이트 13페이지 고양이 카드 수 : 12
4번 사이트 14페이지 고양이 카드 수 : 12
4번 사이트 15페이지 고양이 카드 수 : 12
4번 사이트 16페이지 고양이 카드 수 : 12
4번 사이트 17페이지 고양이 카드 수 : 12
4번 사이트 18페이지 고양이 카드 수 : 12
4번 사이트 19페이지 고양이 카드 수 : 12
4번 사이트 20페이지 고양이 카드 수 : 12
4번 사이트 21페이지 고양이 카드 수 : 12
4번 사이트 22페이지 고양이 카드 수 : 12
4번 사이트 23페이지 고양이 카드 수 : 12
4번 사이트 24페이지 고양이 카드 수 : 12
4번 사이트 25페이지 고양이 카드 수 : 12
4번 사이트 26페이지 고양이 카드 수 : 12
4번 사이트 27페이지 고양이 카드 수 : 12
4번 사이트 28페이지 고양이 카드 수 : 12
4번 사이트 29페이지 고양이 카드 수 : 12
4번 사이트 30페이지 고양이 카드 수 : 12
4번 사이트 31페이지 고양이 카드 수 : 12
4번 사이트 32페이지 고양이 카드 수 : 12
4번 사이트 33페이지 고양이 카드 수 : 12
4번 사이트 34페이지 고양이 카드 수 : 12
4번 사이트 35페이지 고양이 카드 수 : 12
4번 사이트 36페이지 고양이 카드 수 : 12
4번 사이트 37페이지 고양이 카드 수 : 12
4번 사이트 38페